# Validasi Digital Checkup — Fuzzy AHP + TOPSIS
Notebook ini menghitung ulang data canonical yang diekspor dari halaman admin. Semua hasil dibandingkan dengan keluaran web menggunakan toleransi relatif `1e-10`. Tidak ada data contoh yang dibuat di notebook.

In [ ]:
import json, math
from google.colab import files

EPS = 1e-12
TOL = 1e-10
D = [f'C{i}' for i in range(1, 9)]
M = [f'A{i}' for i in range(1, 8)]
RI8 = 1.41

def geo(values):
    return math.exp(sum(math.log(x) for x in values) / len(values))

def normalize(values):
    total = sum(values)
    assert total > EPS
    return [x / total for x in values]

def fuzzy(x):
    if x == 1: return [1., 1., 1.]
    for k in range(2, 10):
        t = [max(1, k - 1), k, min(9, k + 1)]
        if x == k: return t
        if x == 1 / k: return [1 / t[2], 1 / t[1], 1 / t[0]]
    raise ValueError('Skala pairwise harus 1, 2–9, atau resiprokalnya.')

def crisp_matrix(pairs):
    assert len(pairs) == 28, 'Setiap ahli wajib memiliki 28 pasangan.'
    matrix = [[1. for _ in D] for _ in D]
    seen = set()
    for p in pairs:
        i, j = D.index(p['left']), D.index(p['right'])
        assert i < j and (i, j) not in seen
        seen.add((i, j))
        k = p['intensity']
        value = 1 if p['preferred'] == 'equal' else (k if p['preferred'] == p['left'] else 1 / k)
        matrix[i][j], matrix[j][i] = value, 1 / value
    return matrix

def consistency(matrix):
    vector = [1 / 8] * 8
    for iteration in range(10000):
        nxt = normalize([sum(row[j] * vector[j] for j in range(8)) for row in matrix])
        delta = max(abs(nxt[i] - vector[i]) for i in range(8))
        vector = nxt
        if delta <= 1e-14: break
    else: raise RuntimeError('Iterasi eigen tidak konvergen.')
    lm = sum(sum(row[j] * vector[j] for j in range(8)) / vector[i] for i, row in enumerate(matrix)) / 8
    ci = max(0., (lm - 8) / 7)
    return {'lambdaMax': lm, 'CI': ci, 'CR': ci / RI8, 'consistent': ci / RI8 <= .1 + EPS}

def recompute(inp):
    assert inp['dimensionCodes'] == D and inp['moduleCodes'] == M
    experts = inp['experts']
    assert len(experts) == 3, 'Model wajib memakai tepat tiga ahli.'
    crisp = [crisp_matrix(e['pairs']) for e in experts]
    cr = [consistency(matrix) for matrix in crisp]
    assert all(item['consistent'] for item in cr), 'Ada penilaian ahli dengan CR > 0,10.'
    individual_tfn = [[[[*fuzzy(x)] for x in row] for row in matrix] for matrix in crisp]
    group = [[[geo([individual_tfn[e][i][j][k] for e in range(3)]) for k in range(3)] for j in range(8)] for i in range(8)]
    row_gm = [[geo([group[i][j][k] for j in range(8)]) for k in range(3)] for i in range(8)]
    totals = [sum(row[k] for row in row_gm) for k in range(3)]
    fuzzy_weights = [[row[0] / totals[2], row[1] / totals[1], row[2] / totals[0]] for row in row_gm]
    centroids = [sum(row) / 3 for row in fuzzy_weights]
    w = normalize(centroids)
    R = [[sum(e['relevance'][m][d] for e in experts) / 3 for d in D] for m in M]
    answers = inp['answers']
    assert len(answers) == 32 and all(type(v) is int and 0 <= v <= 4 for v in answers.values())
    S = [sum(answers[f'Q{4*j+k+1:02d}'] for k in range(4)) / 16 * 100 for j in range(8)]
    g = [1 - score / 100 for score in S]
    H = sum(w[j] * g[j] for j in range(8))
    result = {'w':w, 'R':R, 'S':S, 'g':g, 'H':H, 'q':[0.]*8, 'C':[], 'rank':[], 'ranking':[], 'individualCR':cr}
    if H <= EPS:
        return {**result, 'status':'NO_IDENTIFIED_GAP'}
    q = [w[j] * g[j] / H for j in range(8)]
    result['q'] = q
    den = [math.sqrt(sum(row[j] ** 2 for row in R)) for j in range(8)]
    if any(den[j] == 0 and g[j] > EPS for j in range(8)):
        return {**result, 'status':'UNCOVERED_DIMENSION'}
    N = [[row[j] / den[j] if den[j] else 0. for j in range(8)] for row in R]
    V = [[row[j] * q[j] for j in range(8)] for row in N]
    high = [max(row[j] for row in V) for j in range(8)]
    low = [min(row[j] for row in V) for j in range(8)]
    dplus = [math.sqrt(sum((row[j] - high[j]) ** 2 for j in range(8))) for row in V]
    dminus = [math.sqrt(sum((row[j] - low[j]) ** 2 for j in range(8))) for row in V]
    if all(dplus[i] + dminus[i] <= EPS for i in range(7)):
        return {**result, 'status':'ALTERNATIVES_NOT_DISTINGUISHABLE'}
    C = [dminus[i] / (dplus[i] + dminus[i]) for i in range(7)]
    rank = [1 + sum(other > value + EPS for other in C) for value in C]
    ranking = sorted([{'code':M[i], 'C':C[i], 'rank':rank[i], 'Dplus':dplus[i], 'Dminus':dminus[i]} for i in range(7)], key=lambda x:(x['rank'], x['code']))
    return {**result, 'C':C, 'rank':rank, 'ranking':ranking, 'status':'RANKED'}

def max_error(actual, expected):
    if isinstance(expected, list): return max([max_error(a, e) for a, e in zip(actual, expected)] or [0.])
    if isinstance(expected, (int, float)): return abs(actual - expected) / max(1., abs(expected))
    return 0. if actual == expected else float('inf')

uploaded = files.upload()
filename = next(iter(uploaded))
bundle = json.loads(uploaded[filename])
assert bundle['schemaVersion'] == 'DL-COLAB-1.0'
record = bundle['records'][0]
assert record['canonicalInput'] and record['run'], 'Pilih UMKM yang hasilnya sudah RANKED.'
actual = recompute(record['canonicalInput'])
expected = record['run']['trace']
print('UMKM:', record['umkmCode'], '| status:', actual['status'])
print('CR ahli:', [round(x['CR'], 10) for x in actual['individualCR']])
print('Top 3:', [(x['code'], x['rank'], round(x['C'], 10)) for x in actual['ranking'][:3]])
for key in ['w', 'R', 'S', 'g', 'H', 'q', 'C', 'rank']:
    error = max_error(actual[key], expected[key])
    print(f'{key:>4}: galat maksimum = {error:.3e}', 'LULUS' if error <= TOL else 'GAGAL')
    assert error <= TOL, f'{key} berbeda lebih dari toleransi {TOL}'
assert [x['code'] for x in actual['ranking']] == [x['code'] for x in expected['ranking']]
print('\nVALIDASI LULUS — hasil web dan notebook sama sampai toleransi 1e-10.')
